# PDX receptor–target ablation study

This notebook runs ten matched fits with receptor–target weights learned across curated database edges and ten fits with equal weights on those same edges. It compares detected-ligand counts, ligand ranks, the prespecified PTN/FAM3C/EFNA5 versus VEGF separation, and the stability of learned receptor–target weights.

PDX expression is rebuilt from raw counts in `.raw.X`: normalize each cell to 10,000 counts, then apply natural-log1p. The original counts are retained in `layers["counts"]`; saved expression and gene-selection statistics are not used.


In [ ]:
from itertools import combinations
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.stats import spearmanr
import xenocomm as xc
from scipy import sparse

sns.set_theme(context="notebook", style="whitegrid")

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "data").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
DATA_DIR = NOTEBOOK_DIR / "data/melanoma_pdx_10k"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs/ablation_10k"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def read_pdx_counts(path):
    stored = ad.read_h5ad(path)
    if stored.raw is None:
        raise ValueError(f"{path} must contain raw counts in .raw.X")
    counts = sparse.csr_matrix(stored.raw.X, copy=True)
    if (
        not np.isfinite(counts.data).all()
        or np.any(counts.data < 0)
        or np.any(counts.data != np.floor(counts.data))
    ):
        raise ValueError(f"{path}: .raw.X must contain nonnegative integer counts")
    if np.any(np.asarray(counts.sum(axis=1)).ravel() <= 0):
        raise ValueError(f"{path}: every cell must have a positive count total")
    data = ad.AnnData(
        X=counts.astype(np.float32),
        obs=stored.obs[["sample", "cell_type"]].copy(),
        var=stored.raw.var[["gene_id"]].copy(),
    )
    data.layers["counts"] = counts
    sc.pp.normalize_total(data, target_sum=10_000)
    sc.pp.log1p(data)
    return data


adata_mouse = read_pdx_counts(DATA_DIR / "adata_mouse.h5ad")
adata_human = read_pdx_counts(DATA_DIR / "adata_human.h5ad")


In [ ]:
SEEDS = (20260716, 20260718, 20260720, 20260722, 20260724, 20260726, 20260728, 20260730, 20260801, 20260803)
MODES = {"Learned": "learned", "Fixed": "fixed"}
STEPS_PER_BATCH = 250
EPOCHS = 5
POSTERIOR_SAMPLES = 2_000
VALIDATION_CELLS = 4096
VALIDATION_SPLIT_SEED = 20260717
VALIDATION_SEED = 20260718


def validation_indices(obs, size, seed):
    strata = obs[["sample", "cell_type"]].astype(str).agg("|".join, axis=1)
    counts = strata.value_counts().sort_index()
    exact = counts.to_numpy() * size / len(obs)
    quotas = np.floor(exact).astype(int)
    quotas[np.argsort(-(exact - quotas), kind="stable")[: size - quotas.sum()]] += 1
    rng = np.random.RandomState(seed)
    held_out = np.concatenate([
        rng.choice(np.flatnonzero(strata.to_numpy() == label), quota, replace=False)
        for label, quota in zip(counts.index, quotas, strict=True)
    ])
    rng.shuffle(held_out)
    return np.setdiff1d(np.arange(len(obs)), held_out), held_out


training_indices, held_out_indices = validation_indices(
    adata_mouse.obs, VALIDATION_CELLS, VALIDATION_SPLIT_SEED
)
training_mouse = adata_mouse[training_indices]
validation_mouse = adata_mouse[held_out_indices]
network = xc.prepare_network(adata_mouse, adata_human, dispersion_cutoff=-10)
ligand_abundance = xc.compute_ligand_abundance(
    adata_mouse,
    adata_human,
    network["ligands"],
    network["human_ligands"],
    network["ligand_receptor_matrix"],
)
print(f"{len(network['ligands']):,} ligands, {len(network['receptors']):,} receptors, {len(network['targets']):,} targets")


In [ ]:
run_tables = []
run_parameters = {}
for variant, mode in MODES.items():
    for seed in SEEDS:
        stem = f"{mode}_seed_{seed}"
        table_path = OUTPUT_DIR / f"{stem}.parquet"
        parameter_path = OUTPUT_DIR / f"{stem}.npz"
        if table_path.exists() and parameter_path.exists():
            table = pd.read_parquet(table_path)
            with np.load(parameter_path, allow_pickle=False) as saved:
                parameters = {name: np.asarray(saved[name]) for name in saved.files}
        else:
            print(f"Training {variant}, seed {seed}")
            model = xc.XenocommModel(
                training_mouse,
                **network,
                mean_ligand=ligand_abundance,
                receptor_target_mode=mode,
                training_seed=seed,
                posterior_seed=seed + 1,
                batch_size=1024,
                steps_per_batch=STEPS_PER_BATCH,
                epochs=EPOCHS,
            )
            model.train(
                validation_mouse=validation_mouse,
                absolute_tolerance=0.001 * 1024,
                patience=3,
                min_evaluations=5,
                validation_seed=VALIDATION_SEED,
            )
            samples = model.sample(POSTERIOR_SAMPLES)
            parameters = model.get_parameters()
            table = xc.ligand_result_table(
                model.ligands,
                samples,
                model.mean_ligand_np,
                model.ligand_receptor_matrix_np,
                xc.get_receptor_sensitivity(parameters),
            )
            table.to_parquet(table_path, index=False)
            np.savez(parameter_path, **parameters)
        table = table.assign(variant=variant, seed=seed)
        run_tables.append(table)
        run_parameters[(variant, seed)] = parameters

ablation_results = pd.concat(run_tables, ignore_index=True)
ablation_results.to_parquet(OUTPUT_DIR / "ligand_results.parquet", index=False)
run_summary = (
    ablation_results.groupby(["variant", "seed"])["called"]
    .sum()
    .rename("detected_ligands")
    .reset_index()
)
run_summary.to_parquet(OUTPUT_DIR / "run_summary.parquet", index=False)
run_summary


In [ ]:
def separation(table):
    indexed = table.set_index(table["ligand"].str.upper())
    percentile = 1 - (indexed["rank"] - 1) / (len(indexed) - 1)
    supported = percentile.loc[["PTN", "FAM3C", "EFNA5"]].median()
    vegf = percentile.loc[["VEGFA", "VEGFB", "VEGFC", "VEGFD"]].max()
    return float(supported - vegf)


receptor_lookup = {gene.casefold(): index for index, gene in enumerate(adata_mouse.var_names)}
receptor_columns = [receptor_lookup[gene.casefold()] for gene in network["receptors"]]
receptor_mean = np.asarray(training_mouse.X[:, receptor_columns].mean(axis=0)).ravel()
edge_scores = np.sqrt(ligand_abundance[1, :, None] * receptor_mean[None, :])
mass_scores = (edge_scores * network["ligand_receptor_matrix"]).sum(axis=1)
mass_action = pd.DataFrame({"ligand": network["ligands"], "score": mass_scores})
mass_action["rank"] = mass_action["score"].rank(method="first", ascending=False).astype(int)
mass_separation = separation(mass_action)

component_rows = []
concordance_rows = []
for seed in SEEDS:
    learned = ablation_results.query("variant == 'Learned' and seed == @seed")
    fixed = ablation_results.query("variant == 'Fixed' and seed == @seed")
    learned_index = learned.set_index("ligand")
    fixed_index = fixed.set_index("ligand")
    learned_calls = set(learned.loc[learned["called"], "ligand"])
    fixed_calls = set(fixed.loc[fixed["called"], "ligand"])
    union = learned_calls | fixed_calls
    concordance_rows.append({
        "seed": seed,
        "rank_spearman": spearmanr(learned_index["rank"], fixed_index.loc[learned_index.index, "rank"]).statistic,
        "top25_overlap": len(set(learned.nsmallest(25, "rank")["ligand"]) & set(fixed.nsmallest(25, "rank")["ligand"])),
        "call_jaccard": len(learned_calls & fixed_calls) / len(union) if union else 1.0,
    })
    learned_separation = separation(learned)
    fixed_separation = separation(fixed)
    component_rows.extend([
        {"component": "Learned receptor–target weights", "seed": seed, "effect": learned_separation - fixed_separation},
        {"component": "Target expression", "seed": seed, "effect": fixed_separation - mass_separation},
    ])

rank_concordance = pd.DataFrame(concordance_rows)
component_effects = pd.DataFrame(component_rows)
rank_concordance.to_parquet(OUTPUT_DIR / "rank_concordance.parquet", index=False)
component_effects.to_parquet(OUTPUT_DIR / "component_effects.parquet", index=False)
display(rank_concordance.describe())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(data=run_summary, x="variant", y="detected_ligands", ax=axes[0], color="#4b9cbe")
sns.stripplot(data=run_summary, x="variant", y="detected_ligands", ax=axes[0], color="black")
axes[0].set(xlabel="", ylabel="Detected ligands", title="Ligand detection across seeds")
sns.boxplot(data=rank_concordance[["rank_spearman", "call_jaccard"]].rename(columns={"rank_spearman": "Rank correlation", "call_jaccard": "Enriched-ligand overlap"}), ax=axes[1], color="#7a9a01")
axes[1].set_xticks([0, 1], ["Rank Spearman", "Detected-set Jaccard"])
axes[1].set(ylabel="Agreement", title="Learned versus fixed database")
plt.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=component_effects, x="component", y="effect", ax=ax, color="#4b9cbe")
sns.stripplot(data=component_effects, x="component", y="effect", ax=ax, color="black")
ax.axhline(0, color="black", linewidth=1)
ax.set(xlabel="", ylabel="Change in ligand-separation margin", title="Ablation component effects")
plt.tight_layout()


In [ ]:
def effective_weights(parameters):
    weights = np.square(parameters["gamma"])
    return weights / np.clip(weights.sum(axis=1, keepdims=True), 1e-6, 1)


top_targets = {}
for seed in SEEDS:
    weights = effective_weights(run_parameters[("Learned", seed)])
    top_targets[seed] = []
    for receptor_index in range(len(network["receptors"])):
        connected = np.flatnonzero(network["receptor_target_matrix"][receptor_index])
        order = np.argsort(-weights[receptor_index, connected])[:20]
        top_targets[seed].append(connected[order])

stability_rows = []
for left_seed, right_seed in combinations(SEEDS, 2):
    for receptor_index, receptor in enumerate(network["receptors"]):
        left = set(top_targets[left_seed][receptor_index])
        right = set(top_targets[right_seed][receptor_index])
        stability_rows.append({
            "left_seed": left_seed,
            "right_seed": right_seed,
            "receptor": receptor,
            "jaccard": len(left & right) / len(left | right),
        })
receptor_target_stability = pd.DataFrame(stability_rows)
receptor_target_stability.to_parquet(OUTPUT_DIR / "receptor_target_stability.parquet", index=False)

fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(receptor_target_stability["jaccard"], bins=20, ax=ax, color="#4b9cbe")
ax.set(xlabel="Top-20 target Jaccard", ylabel="Receptor–seed pairs", title="Learned receptor–target stability")
plt.tight_layout()
print(f"Median top-20 Jaccard: {receptor_target_stability['jaccard'].median():.3f}")
